In [ ]:
import anndata as ad
import decoupler as dc
#import mofax as mfx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests
import statsmodels.stats.multitest as multitest
import re

def clean_name(name):
    # Remove content in parentheses
    name = re.sub(r'\s*\(.*?\)', '', name)
    # Replace underscores with spaces
    name = name.replace('_', ' ')
    # Trim extra whitespace
    name = name.strip().upper()
    return name

## Finding significant factors for the YAP/TAZ TFs and the top 10 drugs from the positive and negative extremes

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

# --- 1. Significant factors ---
PATH_ADATA = "/home/miguel-agromayor-otero/Escritorio/TFM_datos/mofa_adata_30f.h5ad"
df = pd.read_parquet('/home/miguel-agromayor-otero/Escritorio/TFM_datos/lfc_yap')
adata = ad.read_h5ad(PATH_ADATA)
factors = adata.uns["mofa_weights_factors"]
collectri = dc.get_collectri(organism="human")
tfs_of_interest = ["YAP1", "TEAD1", "TEAD2", "TEAD4", "WWTR1"]

W = pd.DataFrame(
    adata.uns["mofa_weights"],
    index=adata.uns["mofa_weights_genes"],
    columns=factors
)
acts, pvals = dc.run_ulm(W.T, net=collectri, source="source", target="target", weight="weight")
acts_sig = acts.where((pvals < 0.05))
results = acts_sig[tfs_of_interest].dropna(how='all')
print(results)

# --- 2. Scores and top/bottom per factor ---
scores = pd.DataFrame(adata.X, index=adata.obs_names, columns=adata.var_names)
scores['drug'] = adata.obs['drug']
scores_avg = scores.groupby('drug')[factors].mean()

# Check for significant differences between drugs at the two extremes of the factors;
# identify which drugs fall in the top and bottom of each factor, so we can later
# look at which drugs show the most negative LFC, which are the ones of interest
top_bottom = {}
for factor in results.index:
    tfs_sig = results.loc[factor].dropna().index.tolist()
    top_bottom[factor] = {
        'top': scores_avg[factor].nlargest(10).index.tolist(),
        'bottom': scores_avg[factor].nsmallest(10).index.tolist(),
        'tfs': tfs_sig
    }

# --- 3. Here we compare the drugs at the positive and negative extremes of each factor
# based on their LFC, to check for significant differences
stats_list = []
for fac, groups in top_bottom.items():
    for tf in groups["tfs"]:
        v_top = df[(df["drug"].isin(groups["top"])) & (df["feature"] == tf)]["value"]
        v_bot = df[(df["drug"].isin(groups["bottom"])) & (df["feature"] == tf)]["value"]
        if len(v_top) > 0 and len(v_bot) > 0:
            stat, p = ttest_ind(v_top, v_bot, equal_var=False)
            stats_list.append({
                "factor": fac, "TF": tf,
                "mean_top": v_top.mean(), "mean_bot": v_bot.mean(),
                "median_top": v_top.median(), "median_bot": v_bot.median(),
                "t_stat": stat, "pval": p
            })

df_stats = pd.DataFrame(stats_list)
df_stats["padj"] = multipletests(df_stats["pval"], method="fdr_bh")[1]
df_stats["diff_mean"] = df_stats["mean_top"] - df_stats["mean_bot"]

# --- Classify each factor-TF pair by the sign of its ULM activity,
# to determine which side of the factor score corresponds to the phenotype of interest
factors_pos = {}
factors_neg = {}
for factor in top_bottom.keys():
    for tf in top_bottom[factor]["tfs"]:
        act_val = results.loc[factor, tf]
        if act_val > 0:
            factors_pos.setdefault(factor, []).append(tf)
        else:
            factors_neg.setdefault(factor, []).append(tf)

# --- 4b. Average the LFC per drug and TF (collapsing replicates),
# to use as input data for the boxplot charts
df_avg = df.groupby(["drug", "feature"])["value"].mean().reset_index()


In [ ]:
# Positive panel
# Filter only factors with at least one significant TF
pos_factors_sig = {}
for factor, tfs in factors_pos.items():
    tfs_sig = []
    for tf in tfs:
        r = df_stats[(df_stats["factor"] == factor) & (df_stats["TF"] == tf)]
        if len(r) > 0 and r["padj"].values[0] < 0.05:
            tfs_sig.append(tf)
    if tfs_sig:
        pos_factors_sig[factor] = tfs_sig

n = len(pos_factors_sig)
ncols = 3
nrows = -(-n // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(10 * ncols, 8 * nrows))
axes_flat = list(axes.flat) if n > 1 else [axes]

for i, (factor, tfs_panel) in enumerate(pos_factors_sig.items()):
    ax = axes_flat[i]
    groups = top_bottom[factor]
    df_f = pd.concat([
        df_avg[df_avg["drug"].isin(groups["top"]) & df_avg["feature"].isin(tfs_panel)].assign(group="Positive extreme"),
        df_avg[df_avg["drug"].isin(groups["bottom"]) & df_avg["feature"].isin(tfs_panel)].assign(group="Negative extreme")
    ])
    ax.set_facecolor("#e8f5e9")
    sns.boxplot(data=df_f, x="feature", y="value", hue="group",
                palette={"Positive extreme": "#d62728", "Negative extreme": "#1f77b4"},
                order=tfs_panel, 
                width=0.9, ax=ax)
    ax.axhline(0, ls="--", color="grey", lw=0.8)
    ax.set_title(factor, fontsize=36, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("LFC" if i % ncols == 0 else "", fontsize=36)
    ax.tick_params(axis="x", rotation=45, labelsize=36)
    ax.tick_params(axis="y", labelsize=36)

    y_max = df_f["value"].quantile(0.98)
    for j, tf in enumerate(tfs_panel):
        r = df_stats[(df_stats["factor"] == factor) & (df_stats["TF"] == tf)]
        p = r["padj"].values[0]
        label = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else None
        if label:
            ax.text(j, y_max * 0.95, label, ha="center", fontsize=24, fontweight="bold")

    if ax.get_legend():
        ax.get_legend().remove()

for k in range(n, nrows * ncols):
    axes_flat[k].set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=30,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig('/home/miguel-agromayor-otero/Escritorio/TFM_datos/Graficos/boxplots_yaptaz_positiva.pdf',
            format='pdf', bbox_inches='tight')
plt.show()


# Negative panel
neg_factors_sig = {}
for factor, tfs in factors_neg.items():
    tfs_sig = []
    for tf in tfs:
        r = df_stats[(df_stats["factor"] == factor) & (df_stats["TF"] == tf)]
        if len(r) > 0 and r["padj"].values[0] < 0.05:
            tfs_sig.append(tf)
    if tfs_sig:
        neg_factors_sig[factor] = tfs_sig

n = len(neg_factors_sig)
nrows = -(-n // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(10 * ncols, 8 * nrows))
axes_flat = list(axes.flat) if n > 1 else [axes]

for i, (factor, tfs_panel) in enumerate(neg_factors_sig.items()):
    ax = axes_flat[i]
    groups = top_bottom[factor]
    df_f = pd.concat([
        df_avg[df_avg["drug"].isin(groups["top"]) & df_avg["feature"].isin(tfs_panel)].assign(group="Positive extreme"),
        df_avg[df_avg["drug"].isin(groups["bottom"]) & df_avg["feature"].isin(tfs_panel)].assign(group="Negative extreme")
    ])
    ax.set_facecolor("#f3e5f5")
    sns.boxplot(data=df_f, x="feature", y="value", hue="group",
                palette={"Positive extreme": "#d62728", "Negative extreme": "#1f77b4"},
                width=0.9, ax=ax)
    ax.axhline(0, ls="--", color="grey", lw=0.8)
    ax.set_title(factor, fontsize=36, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("LFC" if i % ncols == 0 else "", fontsize=36)
    ax.tick_params(axis="x", rotation=45, labelsize=36)
    ax.tick_params(axis="y", labelsize=36)

    y_max = df_f["value"].quantile(0.98)
    for j, tf in enumerate(tfs_panel):
        r = df_stats[(df_stats["factor"] == factor) & (df_stats["TF"] == tf)]
        p = r["padj"].values[0]
        label = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else None
        if label:
            ax.text(j, y_max * 0.95, label, ha="center", fontsize=24, fontweight="bold")

    if ax.get_legend():
        ax.get_legend().remove()

for k in range(n, nrows * ncols):
    axes_flat[k].set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=30,
           bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig('/home/miguel-agromayor-otero/Escritorio/TFM_datos/Graficos/boxplots_yaptaz_negativa.pdf',
            format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
# 1. Only significant factor-TF pairs
df_stats_sig = df_stats[df_stats['padj'] < 0.05]

# 2. For each factor-TF pair, take the drugs from the extreme with the strongest inhibition
candidates = []
for _, row in df_stats_sig.iterrows():
    fac = row["factor"]
    groups = top_bottom[fac]
    # The extreme with the most negative mean = strongest inhibition
    if row["mean_top"] < row["mean_bot"]:
        drugs = groups["top"]
    else:
        drugs = groups["bottom"]
    for d in drugs:
        candidates.append({"drug": d, "factor": fac, "TF": row["TF"]})

df_cand = pd.DataFrame(candidates)

# Unique candidate drugs
unique_candidates = df_cand["drug"].unique()
n_hits = df_cand.groupby("drug").size().rename("n_factor_tf_hits").sort_values(ascending=False)
print(f"Candidate drugs: {len(unique_candidates)}")
print(f"Number of times each drug is repeated: {n_hits}")
print(unique_candidates)
candidate_drugs_moa_results = adata.obs[adata.obs['drug'].isin(df_cand['drug'])][['drug', 'moa-fine']]
candidate_drugs_moa_results.to_excel("/home/miguel-agromayor-otero/Escritorio/resultados_drogas_candidatas.xlsx", index=False)


## Fisher's exact test. To check whether the drugs prioritized by our model are related to drugs that inhibit YAP/TAZ reported in the literature

In [ ]:
import pandas as pd
from scipy.stats import fisher_exact

# hand-curated list from the literature sources already identified
known_modulators_raw = [
    # direct (PMC6162436, Table 2)
    'digitoxin', 'verteporfin', 'flufenamic acid',
    # indirect (PMC6162436, Table 1)
    'dasatinib', 'dobutamine', 'dimethyl fumarate', 'erlotinib',
    'fluvastatin', 'gefitinib', 'losmapimod', 'melatonin',
    'metformin', 'pazopanib', 'trametinib',
    # justified separately (ROCK class / statins)
    'hydroxyfasudil', 'pitavastatin'
]


known_modulators_raw = [clean_name(d) for d in known_modulators_raw]

# my 357 drugs and my 60 prioritized drugs
universe = set(df_avg['drug'].apply(clean_name))
prioritized = set(df_cand['drug'].apply(clean_name))

# only the ones present in my starting universe count
known_modulators = set(known_modulators_raw) & universe

N = len(universe)
K = len(known_modulators)
n = len(prioritized)
a = len(prioritized & known_modulators)

b, c = n - a, K - a  # b -> among my prioritized drugs, those that are NOT known modulators ("new" candidates); c -> known modulators present in the universe that my method did NOT prioritize
d = N - n - c  # the rest — neither prioritized nor known modulators
table = [[a, b], [c, d]]

odds_ratio, p_value = fisher_exact(table, alternative='greater')

print(f"N={N}, K={K}, n={n}, a={a}")
print(f"Expected by chance: {K*(n/N):.2f}")
print(f"Odds ratio: {odds_ratio:.2f}, p = {p_value:.4f}")
print(f"Known modulators in the universe: {known_modulators}")
print(f"Of those, in my prioritized set: {prioritized & known_modulators}")
print(f"{a},{b},{c},{d}")

## Validación a partir de DepMap

In [ ]:
# ## Descarga de los datos de interés para trabajr con DepMap
# import requests
# import os

# prism_article_id = '25917643'
# crispr_article_id = '25880521'
# ruta_salida = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/"

# for article_id, nombres in [
#     (prism_article_id, ['Repurposing_Public_24Q2_LFC.csv',
#                         'Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv',
#                         'Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv',
#                         'Repurposing_Public_24Q2_Treatment_Meta_Data.csv',
#                         'Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv']),
#     (crispr_article_id, ['CRISPRGeneDependency.csv',
#                          ])
# ]:
#     response = requests.get(f'https://api.figshare.com/v2/articles/{article_id}/files')
#     files = response.json()
#     for f in files:
#         if f['name'] in nombres:
#             print(f"Descargando {f['name']}...")
#             r = requests.get(f['download_url'])
#             with open(os.path.join(ruta_salida, f['name']), 'wb') as out:
#                 out.write(r.content)
#             print(f"Guardado: {f['name']}")

In [ ]:
# cell_line_metadata = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Repurposing_Public_24Q2_Cell_Line_Meta_Data.csv")
# metadata = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Repurposing_Public_24Q2_Extended_Primary_Compound_List.csv")
# epdm = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos_yap/Repurposing_Public_24Q2_Extended_Primary_Data_Matrix.csv")
# crispr = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos_yap/CRISPRGeneDependency.csv", index_col=0)
# cell_line_organos = pd.read_csv("/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/depmap/Model.csv")
# print(f"CRISPR: {crispr.shape}, DepMap: {epdm.shape}, metadata : {metadata.shape}" )

In [ ]:
PATH_OUT = '/home/miguel-agromayor-otero/Escritorio/TFM_datos/datos_yap/datos_yap/'

In [ ]:
crispr = pd.read_csv(PATH_OUT + 'CRISPRGeneDependency.csv')
data = pd.read_csv(PATH_OUT + 'drug_cell_lines.csv')
crispr = crispr.set_index('Unnamed: 0')
crispr.index.name = 'ModelID'          

cell_line_organs = pd.read_csv('/home/miguel-agromayor-otero/Model.csv')

collectri = dc.get_collectri(organism="human")
tfs_of_interest = ["YAP1", "TEAD1", "TEAD2", "TEAD4", "WWTR1"]

# Locate the gene columns in the CRISPR matrix
yap_col   = [c for c in crispr.columns if c.startswith("YAP1")][0]
wwtr1_col = [c for c in crispr.columns if c.startswith("WWTR1")][0]
tead_cols = [c for c in crispr.columns if any(c.startswith(t) for t in ["TEAD1", "TEAD2", "TEAD4"])]

yap_mask = (
    (crispr[yap_col] > 0.5) |
    (crispr[wwtr1_col] > 0.5) |
    crispr[tead_cols].gt(0.5).any(axis=1)
)

# use .index to get the ModelIDs (ACH-...), not the entire DataFrame
yap_ids    = pd.DataFrame({'ModelID': crispr[yap_mask].index})
no_yap_ids = pd.DataFrame({'ModelID': crispr[~yap_mask].index})

cols = ['ModelID', 'OncotreeLineage', 'OncotreeSubtype', 'OncotreePrimaryDisease',
        'LegacyMolecularSubtype', 'LegacySubSubtype']

yap_dependent_cell_lines   = yap_ids.merge(cell_line_organs[cols], on='ModelID', how='inner')
yap_independent_cell_lines = no_yap_ids.merge(cell_line_organs[cols], on='ModelID', how='inner')

print(f"YAP-dependent: {len(yap_dependent_cell_lines)}")
print(f"YAP-independent: {len(yap_independent_cell_lines)}")

In [ ]:
# # Preparar tabla de DepMap en formato largo: depmap_id x BRD_ID → LFC
# epm = epdm.rename(columns={'Unnamed: 0': 'BRD_ID'}).set_index('BRD_ID')
# rep_clean = metadata[['IDs', 'Drug.Name', 'MOA', 'Synonyms']].drop_duplicates('IDs').rename(columns={'IDs': 'BRD_ID'})
# epm_t = epm.T
# epm_t.index.name = 'depmap_id' 
# epm_t = epm_t.reset_index().merge(cell_line_metadata[['depmap_id', 'ccle_name']], on='depmap_id', how='left')
# epm_t['cell_line'] = epm_t['ccle_name'].str.split('_').str[0] 
# # Pasar a formato largo excluyendo columnas de metadata
# brd_cols = [c for c in epm_t.columns if c not in ['depmap_id', 'ccle_name', 'cell_line', 'index']]
# epm_long = epm_t.melt(id_vars=['depmap_id', 'cell_line'], value_vars=brd_cols, var_name='BRD_ID', value_name='LFC').dropna(subset=['LFC'])
# epm_final = epm_long.merge(rep_clean[['BRD_ID', 'Drug.Name','MOA']], on='BRD_ID').drop_duplicates()

In [ ]:
# #  Compruebobación la intersección real
# ids_epm = set(epm_final['depmap_id'].unique())
# ids_dep = set(lineas_cel_yap_dependent['ModelID'].unique())
# ids_indep = set(lineas_cel_yap_indep['ModelID'].unique())

# print(f"Total líneas en epm_final: {len(ids_epm)}")
# print(f"Total líneas YAP-dep: {len(ids_dep)}")
# print(f"Total líneas YAP-indep: {len(ids_indep)}")
# print(f"YAP-dep en epm_final: {len(ids_epm & ids_dep)}")
# print(f"YAP-indep en epm_final: {len(ids_epm & ids_indep)}")

In [ ]:
dep_lines_per_primary = yap_dependent_cell_lines.groupby('OncotreePrimaryDisease')['ModelID'].apply(list).to_dict()
indep_lines_per_primary = yap_independent_cell_lines.groupby('OncotreePrimaryDisease')['ModelID'].apply(list).to_dict()
print(dep_lines_per_primary)
print(indep_lines_per_primary)

# Count lines per tissue in each group
count_dep   = yap_dependent_cell_lines.groupby('OncotreePrimaryDisease')['ModelID'].count().rename('ydep')
count_indep = yap_independent_cell_lines.groupby('OncotreePrimaryDisease')['ModelID'].count().rename('nodep')

subtype_summary = pd.DataFrame({'ydep': count_dep, 'nodep': count_indep}).dropna()

well_powered_subtypes = subtype_summary[
    (subtype_summary['ydep'] >= 15) &
    (subtype_summary['nodep'] >= 15)
].index

print(f"Tissues with n≥15 in both groups: {len(well_powered_subtypes)}")
print(subtype_summary.loc[well_powered_subtypes])

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import re


data['Drug.Name.clean'] = data['Drug.Name'].apply(clean_name)
mofa_drugs = [clean_name(d) for d in unique_candidates]

rlm_results = []
n_failures = 0

for subtype in well_powered_subtypes:
    for drug in mofa_drugs:
        brd_ids = data[data['Drug.Name.clean'] == clean_name(drug)]['BRD_ID'].tolist()
        if not brd_ids:
            continue

        df_drug = data[data['BRD_ID'].isin(brd_ids)].copy()
        df_drug['yap_dep'] = df_drug['depmap_id'].isin(
            yap_dependent_cell_lines['ModelID']
        ).astype(int)

        df_drug = df_drug.merge(
            cell_line_organs[['ModelID', 'OncotreePrimaryDisease']],
            left_on='depmap_id', right_on='ModelID', how='inner'
        )
        df_drug = df_drug[df_drug['OncotreePrimaryDisease'] == subtype]

        lfc_dep   = df_drug[df_drug['yap_dep'] == 1]['LFC'].values
        lfc_nodep = df_drug[df_drug['yap_dep'] == 0]['LFC'].values
        n_dep   = df_drug[df_drug['yap_dep'] == 1]['depmap_id'].nunique()
        n_nodep = df_drug[df_drug['yap_dep'] == 0]['depmap_id'].nunique()

        # --- Selectivity index (Jose's comment) ---
        median_dep = np.median(lfc_dep) if len(lfc_dep) > 0 else np.nan
        median_nodep = np.median(lfc_nodep) if len(lfc_nodep) > 0 else np.nan
        difference = median_dep - median_nodep
        epsilon = 0.1
        selectivity_index = difference / (abs(median_nodep) + epsilon)
        # -----------------------------------------------------

        try:
            fit = smf.rlm('LFC ~ yap_dep', data=df_drug).fit()
            ci = fit.conf_int()

            rlm_results.append({
                'subtype':             subtype,
                'Drug.Name':           drug,
                'Intercept':           fit.params['Intercept'],
                'coef':                fit.params['yap_dep'],
                'ci_low':              ci.loc['yap_dep'][0],
                'ci_high':             ci.loc['yap_dep'][1],
                'pval':                fit.pvalues['yap_dep'],
                'n_dep':               n_dep,
                'n_nodep':             n_nodep,
                'median_dep':          median_dep,
                'median_nodep':        median_nodep,
                'selectivity_index':   selectivity_index,
            })
        except Exception as e:
            n_failures += 1
            continue

print(f"Fitted models: {len(rlm_results)} | Failed/discarded: {n_failures}")

res_lm = pd.DataFrame(rlm_results)
res_lm['padj'] = res_lm.groupby('subtype')['pval'].transform(
    lambda x: multipletests(x, method='fdr_bh')[1]
)

final_candidates = res_lm[(res_lm['padj'] < 0.05) & (res_lm['coef'] < 0)].sort_values('coef')
print(f"Candidates: {len(final_candidates)}")
final_candidates[['subtype', 'Drug.Name', 'coef', 'padj', 'median_dep', 'median_nodep', 'selectivity_index']]

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

n = len(final_candidates)
fig, axes = plt.subplots(1, n, figsize=(20 * n, 24), squeeze=False)
axes = axes[0]
fig.subplots_adjust(top=0.85)
for ax, (_, row) in zip(axes, final_candidates.iterrows()):
    # Rebuild the drug data for this subtype
    brd_ids = data[data['Drug.Name.clean'] == clean_name(row['Drug.Name'])]['BRD_ID'].tolist()
    df_drug = data[data['BRD_ID'].isin(brd_ids)].copy()
    df_drug['yap_dep'] = df_drug['depmap_id'].isin(
        yap_dependent_cell_lines['ModelID']
    ).astype(int)
    df_drug = df_drug.merge(
        cell_line_organs[['ModelID', 'OncotreePrimaryDisease']],
        left_on='depmap_id', right_on='ModelID', how='inner'
    )
    df_drug = df_drug[df_drug['OncotreePrimaryDisease'] == row['subtype']]
    df_drug = df_drug.dropna(subset=['LFC'])

    lfc_nodep = df_drug[df_drug['yap_dep'] == 0]['LFC'].values
    lfc_dep = df_drug[df_drug['yap_dep'] == 1]['LFC'].values

    bp = ax.boxplot(
        [lfc_nodep, lfc_dep],
        positions=[1, 2],
        widths=0.6,
        patch_artist=True,
        medianprops=dict(color='black', linewidth=1),
        flierprops=dict(marker='o', markersize=2, alpha=0.4),
    )

    colors = ['#8593a3', '#c0392b']
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_edgecolor('black')
        patch.set_linewidth(0.8)

    ax.axhline(0, ls='--', color='gray', linewidth=0.8, zorder=0)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(
        [f"Independent\n(n={row['n_nodep']})",
         f"Dependent\n(n={row['n_dep']})"],
        fontsize=46
    )
    ax.tick_params(axis='y', labelsize=46)
    ax.set_title(f"{row['Drug.Name']} ({row['subtype']})", fontsize=46, fontweight='bold')
    ax.text(0.5, 0.99,
            f"coef = {row['coef']:.2f}  · padj = {row['padj']:.1e} · 95% CI [{row['ci_low']:.2f}, {row['ci_high']:.2f}]",
            transform=ax.transAxes, ha='center', va='top',
            fontsize=36, color='#333333')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel('Log Fold Change (LFC)', fontsize=46)
plt.tight_layout(rect=[0, 0, 1, 0.85])
plt.savefig("/home/miguel-agromayor-otero/Escritorio/TFM_datos/Graficos/candidatas.png", bbox_inches='tight')